In [ ]:
#!pip install numpy   # install numpy

#!pip install pandas matplotlib seaborn scikit-learn   # install multiple libraries at once

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip install gspread gspread-dataframe   # install to connect to google sheets

print("Thank you")

In [ ]:
# copied export code from google sheets "sheets to collab" extension

import gspread
import pandas as pd
from google.auth import default
from google.colab import auth, userdata
spreadsheet_id = userdata.get('SPREADSHEET_ID')

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
# Pull spreadsheet ID from Colab Secrets
spreadsheet = gc.open_by_key(spreadsheet_id)
worksheet = spreadsheet.get_worksheet(0)
df = pd.DataFrame(worksheet.get())
# Code for making the first row as header. Remove if not needed.
df.columns = df.iloc[0]
df = df.drop(0)
df.head()

In [ ]:
insight = gc.open('Tutor Matching Public')   # open & rename spreadsheet

# open & rename specific worksheets in spreadsheet
students = insight.worksheet('Students')

studentsdf = pd.DataFrame(students.get_all_values())
#studentsdf.head()  # I see here that the first row of data is actually the columns.
# To fix:
studentsdf.columns = studentsdf.iloc[0]
studentsdf = studentsdf.iloc[1:].reset_index(drop=True)

studentsdf.head()

In [ ]:
# open & rename specific worksheets in spreadsheet
tutors = insight.worksheet('Tutors')

tutorsdf = pd.DataFrame(tutors.get_all_values())

tutorsdf.columns = tutorsdf.iloc[0]
tutorsdf = tutorsdf.iloc[1:].reset_index(drop=True)

tutorsdf.head()

In [ ]:
# open & rename specific worksheets in spreadsheet
assignments = insight.worksheet('Assignments')

assignmentsdf = pd.DataFrame(assignments.get_all_values())

assignmentsdf.columns = assignmentsdf.iloc[0]
assignmentsdf = assignmentsdf.iloc[1:].reset_index(drop=True)

assignmentsdf.head()

In [ ]:
# I want to check all of my dataframes
print(studentsdf.columns)
print(studentsdf.shape)

In [ ]:
print(tutorsdf.columns)
print(tutorsdf.shape)

In [ ]:
print(assignmentsdf.columns)
print(assignmentsdf.shape)

In [ ]:
# For every assignment, attach the student information
matches = assignmentsdf.merge(
    studentsdf,
    on="Student ID",
    how="left"
)

# For every assignment, also attach the tutor information
matches = matches.merge(
    tutorsdf,
    on="Tutor ID",
    how="left"
)

print(matches.columns)
print(matches.shape)

In [ ]:
# It does look like we retained all our columns but only have 18 records

# studentsdf should now have all the students with assignments
matches[["Student ID", "Student Name"]]

In [ ]:
# We want to create Student Clusters for comparison, and I want to use NLP to
# create one "profile" to analyze each student.

# fillna to clean students data

studentsdf["StudentProfile"] = (
    studentsdf["Notes"].fillna("") + " " +
    studentsdf["Calendly Recap"].fillna("") + " " +
    studentsdf["Areas for Growth"].fillna("") + " " +
    studentsdf["Learning Style"].fillna("") + " " +
    studentsdf["Personality"].fillna("") + " " +
    studentsdf["Special Needs"].fillna("") + " " +
    studentsdf["Personal Interests"].fillna("")
)

studentsdf[["Student Name", "StudentProfile"]].sample(1)

In [ ]:
# Give me a list of the students who did NOT get joined bc they did not have an assignment

studentsdf["Student ID"].isin(assignmentsdf["Student ID"])

studentsb = studentsdf[
    ~studentsdf["Student ID"].isin(assignmentsdf["Student ID"])
].copy()

studentsb[["Student ID", "Student Name","StudentProfile"]]

In [ ]:
# Clean tutors data

tutorsdf["TutorProfile"] = (
    tutorsdf["Expertise"].fillna("") + " " +
    tutorsdf["Special Skills"].fillna("") + " " +
    tutorsdf["Interests"].fillna("") + " " +
    tutorsdf["Introductions"].fillna("")
)

tutorsdf[["Tutor Name", "TutorProfile"]].sample(1)

#NLP   TF-IDF

##Term Frequency-Inverse Document Frequency


In [ ]:
# We'll find the most important words (relatively) for each profile and
# compare student and tutor descriptions (using similarity measures later)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=500
)


student_vectors = vectorizer.fit_transform(
    studentsdf["StudentProfile"]
)

student_vectors.shape

In [ ]:
vectorizer.get_feature_names_out()

In [ ]:
import pandas as pd

student_tfidf = pd.DataFrame(
    student_vectors.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=studentsdf["Student Name"]
)

student_tfidf.head()

In [ ]:
tutor_vectors = vectorizer.transform(
    tutorsdf["TutorProfile"]
)

tutor_vectors.shape

In [ ]:
tutor_tfidf = pd.DataFrame(
    tutor_vectors.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=tutorsdf["Tutor Name"]
)

tutor_tfidf.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compare every student to every tutor
similarity = cosine_similarity(
    student_vectors,
    tutor_vectors
)

similarity.shape

Cool.

In [ ]:
# Turn the matrix into a dataframe

similarity_df = pd.DataFrame(
    similarity,
    index=studentsdf["Student Name"],
    columns=tutorsdf["Tutor Name"]
)

similarity_df.head()

In [ ]:
# Quick check on what this looks like for one student

similarity_df.loc["STE"].sort_values(ascending=False)

This answers the question, "Based on the information in their profiles, which tutors look most similar to what this student needs?"

In [ ]:
# Apply to studentsb
studentsb_vectors = vectorizer.transform(
    studentsb["StudentProfile"]
)
studentsb_similarity = cosine_similarity(
    studentsb_vectors,
    tutor_vectors
)
studentsb_similarity_df = pd.DataFrame(
    studentsb_similarity,
    index=studentsb["Student Name"],
    columns=tutorsdf["Tutor Name"]
)

studentsb_similarity_df.head()

In [ ]:
similarity_df.loc["STZ"].sort_values(ascending=False)

In [ ]:
# So this gives me similarity for all students and tutors:
print(similarity_df)

In [ ]:
# and this gives me similarity for all unmatched students:
print(studentsb_similarity_df)

In [ ]:
# Top 3 tutors for every unmatched student (studentsb)
top3_studentsb = (
    studentsb_similarity_df
    .apply(lambda row: row.sort_values(ascending=False).head(3), axis=1)
)

top3_studentsb

In [ ]:
def get_top_tutors(similarity_dataframe, top_n=3):
    recommendations = []

    for student_name, row in similarity_dataframe.iterrows():
        top_tutors = row.sort_values(ascending=False).head(top_n)

        for rank, (tutor_name, score) in enumerate(top_tutors.items(), start=1):
            recommendations.append({
                "Student Name": student_name,
                "Rank": rank,
                "Tutor Name": tutor_name,
                "Similarity Score": score
            })

    return pd.DataFrame(recommendations)

top3_all_students = get_top_tutors(similarity_df, top_n=3)
top3_studentsb = get_top_tutors(studentsb_similarity_df, top_n=3)

top3_all_students.head(15)

In [ ]:
def top_tutors_wide(similarity_df, top_n=3):
    rows = []

    for student, scores in similarity_df.iterrows():
        top = scores.sort_values(ascending=False).head(top_n)

        row = {"Student": student}

        for i, tutor in enumerate(top.index, start=1):
            row[f"{i}st Match" if i == 1 else f"{i}nd Match" if i == 2 else f"{i}rd Match"] = tutor

        rows.append(row)

    return pd.DataFrame(rows)

top3 = top_tutors_wide(similarity_df)

top3.head()

In [ ]:
print(top3)

In [ ]:
top3_unmatched = top_tutors_wide(studentsb_similarity_df)

top3_unmatched